# Observability with LangSmith for Telecom Agents
## Tracing, Logging, Replay, Cost Tracking, and LangSmith Trace View
### Step-by-step hands-on with LangGraph, LangSmith, MCP concepts, and Telecom tools

This notebook is a **beginner-friendly, step-by-step lab** that shows how to add **observability** to a telecom agent workflow.

You will learn:

1. What observability means for AI agents
2. What LangSmith tracing is
3. How traces, runs, logging, and replay work
4. What cost tracking means
5. How LangSmith trace view helps debugging
6. How to enable tracing
7. How to build a small telecom LangGraph workflow
8. How to instrument tools and inspect what happened

---

## What you will build

A telecom agent flow for a query like:

> "My mobile internet is slow in Pune. What should support do next?"

The flow will:
- fetch customer profile
- check outage
- search telecom SOP / RCA context
- create a support response

We will also add:
- tracing hooks
- structured logs
- replay-friendly state capture
- token / cost placeholders
- MCP-style tool catalog concepts

## Why observability matters

Without observability, if your agent gives a wrong answer, you only see the final output.

With observability, you can see:
- which tool was called
- what input it received
- what output it returned
- what the graph state looked like
- where the failure happened
- how much the run cost

LangSmith is LangChain’s observability platform for tracing, monitoring, and debugging LLM applications, and its docs describe traces as end-to-end execution records made of runs. It also supports tracing LangGraph applications and cost tracking. Official docs also show that tracing can be enabled with environment variables and that the UI lets you inspect traces. 

In [ ]:
# Uncomment if needed in a fresh environment
# %pip install -U langgraph pydantic
# Optional if you want to send real traces to LangSmith:
# %pip install -U langsmith langchain-core

## Step 1 — Imports

In [1]:
from __future__ import annotations

from typing import TypedDict, Dict, Any, Optional, List
from dataclasses import dataclass
from pprint import pprint
from datetime import datetime
import json
import os

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

## Step 2 — Beginner concepts

### A. Trace
A **trace** is the full story of one request from start to finish.

### B. Run
A **run** is one step inside that trace.

Example:
- trace = whole telecom incident workflow
- run 1 = get customer profile
- run 2 = check outage
- run 3 = search SOP
- run 4 = generate response

### C. Logging
Logging means recording useful details while the workflow runs.

### D. Replay
Replay means re-running or re-inspecting the same workflow inputs and steps to debug what happened.

### E. Cost tracking
Cost tracking means recording how much the run consumed:
- model calls
- tokens
- tool usage
- latency

## Step 3 — MCP basics

MCP stands for **Model Context Protocol**.

For this notebook, the most important beginner idea is:

- **tools** = callable functions
- **resources** = contextual data
- **prompts** = reusable prompt/workflow templates

We will not run a real MCP server here, but we will use an **MCP-style tool catalog** so the workflow is easy to understand and presentation-friendly.

## Step 4 — Tool schemas and contracts

We define contracts for each telecom tool. This makes tracing and debugging easier because the inputs and outputs are structured.

In [2]:
class CustomerProfileInput(BaseModel):
    customer_id: str = Field(..., description="Unique telecom customer ID")

class CustomerProfileOutput(BaseModel):
    customer_id: str
    name: str
    city: str
    plan: str
    network_type: str
    recent_complaints: int

class OutageCheckInput(BaseModel):
    city: str
    issue_type: str

class OutageCheckOutput(BaseModel):
    city: str
    issue_type: str
    outage_detected: bool
    planned_maintenance: bool
    summary: str

class PolicySearchInput(BaseModel):
    query_text: str
    city_filter: Optional[str] = None
    issue_filter: Optional[str] = None

class PolicySearchOutput(BaseModel):
    top_match: str
    source: str

class ResponseInput(BaseModel):
    customer_name: str
    city: str
    diagnosis: str
    next_action: str

class ResponseOutput(BaseModel):
    response_text: str
    prompt_tokens: int
    completion_tokens: int
    estimated_cost_usd: float

## Step 5 — MCP-style tool catalog

This is only a conceptual catalog for learning and presentation. It shows how tools can be described in a structured way.

In [3]:
tool_catalog = [
    {
        "name": "get_customer_profile",
        "description": "Fetch telecom customer profile from CRM",
        "input_schema": CustomerProfileInput.model_json_schema(),
        "output_schema": CustomerProfileOutput.model_json_schema(),
    },
    {
        "name": "check_outage",
        "description": "Check telecom outage by city and issue type",
        "input_schema": OutageCheckInput.model_json_schema(),
        "output_schema": OutageCheckOutput.model_json_schema(),
    },
    {
        "name": "search_policy",
        "description": "Search telecom SOP and RCA guidance",
        "input_schema": PolicySearchInput.model_json_schema(),
        "output_schema": PolicySearchOutput.model_json_schema(),
    },
    {
        "name": "generate_response",
        "description": "Generate customer-friendly telecom response",
        "input_schema": ResponseInput.model_json_schema(),
        "output_schema": ResponseOutput.model_json_schema(),
    },
]

pprint(tool_catalog[0])

{'description': 'Fetch telecom customer profile from CRM',
 'input_schema': {'properties': {'customer_id': {'description': 'Unique '
                                                                'telecom '
                                                                'customer ID',
                                                 'title': 'Customer Id',
                                                 'type': 'string'}},
                  'required': ['customer_id'],
                  'title': 'CustomerProfileInput',
                  'type': 'object'},
 'name': 'get_customer_profile',
 'output_schema': {'properties': {'city': {'title': 'City', 'type': 'string'},
                                  'customer_id': {'title': 'Customer Id',
                                                  'type': 'string'},
                                  'name': {'title': 'Name', 'type': 'string'},
                                  'network_type': {'title': 'Network Type',
                          

## Step 6 — Mock telecom data

We use simple in-memory data so the notebook stays easy to run.

In [5]:
CUSTOMERS = {
    "CUST_1001": {
        "customer_id": "CUST_1001",
        "name": "Ravi Patil",
        "city": "Pune",
        "plan": "Premium 5G",
        "network_type": "5G",
        "recent_complaints": 2,
    },
    "CUST_2001": {
        "customer_id": "CUST_2001",
        "name": "Meera Shah",
        "city": "Mumbai",
        "plan": "Standard 4G",
        "network_type": "4G",
        "recent_complaints": 0,
    },
}

OUTAGES = [
    {
        "city": "Pune",
        "issue_type": "slow_internet",
        "outage_detected": False,
        "planned_maintenance": False,
        "summary": "No citywide outage found. Congestion patterns appear during peak hours."
    },
    {
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "outage_detected": True,
        "planned_maintenance": False,
        "summary": "Packet loss incident active in Mumbai region."
    },
]

POLICIES = [
    {
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If tower utilization exceeds 90 percent during peak hours, classify as likely congestion and notify NOC."
    },
    {
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "content": "If packet loss exceeds 5 percent, investigate transport path degradation and escalate to network operations."
    },
    {
        "city": "Pune",
        "issue_type": "device_issue",
        "content": "Check APN settings, battery saver mode, and network reset status before escalation."
    },
]

## Step 7 — A simple in-notebook tracer

LangSmith is the real observability platform, but for learning purposes we first create a tiny local tracer.

This helps you understand what a trace contains:
- trace id
- runs
- timestamps
- inputs
- outputs
- logs
- estimated cost

In [6]:
TRACE_STORE = []

def start_trace(trace_name: str, initial_input: Dict[str, Any]) -> Dict[str, Any]:
    trace = {
        "trace_id": f"trace-{len(TRACE_STORE) + 1}",
        "trace_name": trace_name,
        "started_at": datetime.utcnow().isoformat(),
        "initial_input": initial_input,
        "runs": [],
        "logs": [],
        "cost_summary": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "estimated_cost_usd": 0.0,
        }
    }
    TRACE_STORE.append(trace)
    return trace

def log_run(trace: Dict[str, Any], node_name: str, node_input: Dict[str, Any], node_output: Dict[str, Any]):
    trace["runs"].append({
        "node": node_name,
        "timestamp": datetime.utcnow().isoformat(),
        "input": node_input,
        "output": node_output,
    })

def log_message(trace: Dict[str, Any], message: str):
    trace["logs"].append({
        "timestamp": datetime.utcnow().isoformat(),
        "message": message,
    })

def add_cost(trace: Dict[str, Any], prompt_tokens: int, completion_tokens: int, cost: float):
    trace["cost_summary"]["prompt_tokens"] += prompt_tokens
    trace["cost_summary"]["completion_tokens"] += completion_tokens
    trace["cost_summary"]["estimated_cost_usd"] += cost

## Step 8 — Telecom tool functions

These are the tools our telecom agent will use.

In [7]:
def get_customer_profile_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = CustomerProfileInput(**payload)
    data = CUSTOMERS[inp.customer_id]
    return CustomerProfileOutput(**data).model_dump()

def check_outage_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = OutageCheckInput(**payload)
    match = next(
        (
            row for row in OUTAGES
            if row["city"] == inp.city and row["issue_type"] == inp.issue_type
        ),
        {
            "city": inp.city,
            "issue_type": inp.issue_type,
            "outage_detected": False,
            "planned_maintenance": False,
            "summary": "No known outage found."
        }
    )
    return OutageCheckOutput(**match).model_dump()

def search_policy_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = PolicySearchInput(**payload)
    candidates = [
        p for p in POLICIES
        if (inp.city_filter is None or p["city"] == inp.city_filter)
        and (inp.issue_filter is None or p["issue_type"] == inp.issue_filter)
    ]
    if candidates:
        return PolicySearchOutput(
            top_match=candidates[0]["content"],
            source="policy_store"
        ).model_dump()
    return PolicySearchOutput(
        top_match="No matching telecom policy found.",
        source="policy_store"
    ).model_dump()

def generate_response_tool(payload: Dict[str, Any]) -> Dict[str, Any]:
    inp = ResponseInput(**payload)
    text = (
        f"Hello {inp.customer_name}, based on initial analysis for {inp.city}, "
        f"the likely diagnosis is: {inp.diagnosis}. Next action: {inp.next_action}."
    )
    # Simulated token and cost numbers for teaching
    prompt_tokens = 120
    completion_tokens = 45
    estimated_cost_usd = 0.0018
    return ResponseOutput(
        response_text=text,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        estimated_cost_usd=estimated_cost_usd,
    ).model_dump()

## Step 9 — Quick manual tool tests

In [8]:
pprint(get_customer_profile_tool({"customer_id": "CUST_1001"}))
pprint(check_outage_tool({"city": "Pune", "issue_type": "slow_internet"}))
pprint(search_policy_tool({"query_text": "slow internet in Pune", "city_filter": "Pune", "issue_filter": "slow_internet"}))

{'city': 'Pune',
 'customer_id': 'CUST_1001',
 'name': 'Ravi Patil',
 'network_type': '5G',
 'plan': 'Premium 5G',
 'recent_complaints': 2}
{'city': 'Pune',
 'issue_type': 'slow_internet',
 'outage_detected': False,
 'planned_maintenance': False,
 'summary': 'No citywide outage found. Congestion patterns appear during peak '
            'hours.'}
{'source': 'policy_store',
 'top_match': 'If tower utilization exceeds 90 percent during peak hours, '
              'classify as likely congestion and notify NOC.'}


## Step 10 — LangGraph state

This state keeps everything the workflow needs:
- user input
- tool outputs
- diagnosis
- final response
- trace id
- replay data

In [12]:
class TelecomObsState(TypedDict, total=False):
    customer_id: str
    user_query: str
    issue_type: str

    customer_profile: Dict[str, Any]
    outage_info: Dict[str, Any]
    policy_info: Dict[str, Any]

    diagnosis: str
    next_action: str
    response: Dict[str, Any]

    trace_id: str
    replay_snapshot: Dict[str, Any]

## Step 11 — Graph nodes with local tracing

Each node will:
1. read the current state
2. call a tool or compute a result
3. log a run into the trace store
4. return partial state update

In [13]:
TRACE_CONTEXT = {"current_trace": None}

def get_profile_node(state: TelecomObsState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current_trace"]
    node_input = {"customer_id": state["customer_id"]}
    profile = get_customer_profile_tool(node_input)
    log_run(trace, "get_profile", node_input, profile)
    return {"customer_profile": profile}

def check_outage_node(state: TelecomObsState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current_trace"]
    node_input = {
        "city": state["customer_profile"]["city"],
        "issue_type": state["issue_type"],
    }
    outage = check_outage_tool(node_input)
    log_run(trace, "check_outage", node_input, outage)
    return {"outage_info": outage}

def search_policy_node(state: TelecomObsState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current_trace"]
    node_input = {
        "query_text": state["user_query"],
        "city_filter": state["customer_profile"]["city"],
        "issue_filter": state["issue_type"],
    }
    policy = search_policy_tool(node_input)
    log_run(trace, "search_policy", node_input, policy)
    return {"policy_info": policy}

def diagnose_node(state: TelecomObsState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current_trace"]

    if state["outage_info"]["outage_detected"]:
        diagnosis = f"Confirmed outage: {state['outage_info']['summary']}"
        next_action = "Escalate to NOC and communicate outage status to the customer."
    else:
        diagnosis = state["policy_info"]["top_match"]
        next_action = "Follow SOP guidance and monitor repeated complaints."

    node_input = {
        "outage_info": state["outage_info"],
        "policy_info": state["policy_info"],
    }
    node_output = {"diagnosis": diagnosis, "next_action": next_action}
    log_run(trace, "diagnose", node_input, node_output)
    return node_output

def response_node(state: TelecomObsState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current_trace"]
    node_input = {
        "customer_name": state["customer_profile"]["name"],
        "city": state["customer_profile"]["city"],
        "diagnosis": state["diagnosis"],
        "next_action": state["next_action"],
    }
    response = generate_response_tool(node_input)
    log_run(trace, "generate_response", node_input, response)
    add_cost(
        trace,
        response["prompt_tokens"],
        response["completion_tokens"],
        response["estimated_cost_usd"],
    )
    return {"response": response}

## Step 12 — Build the LangGraph workflow

In [14]:
builder = StateGraph(TelecomObsState)

builder.add_node("get_profile", get_profile_node)
builder.add_node("check_outage", check_outage_node)
builder.add_node("search_policy", search_policy_node)
builder.add_node("diagnose", diagnose_node)
builder.add_node("generate_response", response_node)

builder.set_entry_point("get_profile")
builder.add_edge("get_profile", "check_outage")
builder.add_edge("check_outage", "search_policy")
builder.add_edge("search_policy", "diagnose")
builder.add_edge("diagnose", "generate_response")
builder.add_edge("generate_response", END)

graph = builder.compile()
print("Graph compiled")

Graph compiled


## Step 13 — Run a traced telecom workflow

This simulates what LangSmith tracing helps you see in the UI.

In [15]:
initial_state = {
    "customer_id": "CUST_1001",
    "user_query": "My mobile internet is slow in Pune. What should support do next?",
    "issue_type": "slow_internet",
}

trace = start_trace("telecom_incident_workflow", initial_state)
TRACE_CONTEXT["current_trace"] = trace

result = graph.invoke(initial_state)

trace["finished_at"] = datetime.utcnow().isoformat()
trace["final_output"] = result

print("Final response:")
pprint(result["response"])

Final response:
{'completion_tokens': 45,
 'estimated_cost_usd': 0.0018,
 'prompt_tokens': 120,
 'response_text': 'Hello Ravi Patil, based on initial analysis for Pune, the '
                  'likely diagnosis is: If tower utilization exceeds 90 '
                  'percent during peak hours, classify as likely congestion '
                  'and notify NOC.. Next action: Follow SOP guidance and '
                  'monitor repeated complaints..'}


## Step 14 — View the local trace

This is our beginner-friendly local trace view.
It is conceptually similar to how a LangSmith trace helps you inspect:
- each run
- each tool call
- inputs and outputs
- overall cost

In [16]:
pprint(trace)

{'cost_summary': {'completion_tokens': 45,
                  'estimated_cost_usd': 0.0018,
                  'prompt_tokens': 120},
 'final_output': {'customer_id': 'CUST_1001',
                  'customer_profile': {'city': 'Pune',
                                       'customer_id': 'CUST_1001',
                                       'name': 'Ravi Patil',
                                       'network_type': '5G',
                                       'plan': 'Premium 5G',
                                       'recent_complaints': 2},
                  'diagnosis': 'If tower utilization exceeds 90 percent during '
                               'peak hours, classify as likely congestion and '
                               'notify NOC.',
                  'issue_type': 'slow_internet',
                  'next_action': 'Follow SOP guidance and monitor repeated '
                                 'complaints.',
                  'outage_info': {'city': 'Pune',
                      

## Step 15 — Trace view summary

Let's create a cleaner trace summary that looks more like a simple observability dashboard.

In [17]:
summary = {
    "trace_id": trace["trace_id"],
    "trace_name": trace["trace_name"],
    "num_runs": len(trace["runs"]),
    "cost_summary": trace["cost_summary"],
    "nodes_executed": [r["node"] for r in trace["runs"]],
}

pprint(summary)

{'cost_summary': {'completion_tokens': 45,
                  'estimated_cost_usd': 0.0018,
                  'prompt_tokens': 120},
 'nodes_executed': ['get_profile',
                    'check_outage',
                    'search_policy',
                    'diagnose',
                    'generate_response'],
 'num_runs': 5,
 'trace_id': 'trace-1',
 'trace_name': 'telecom_incident_workflow'}


## Step 16 — Logging examples

Logging is useful for custom notes that are not just tool outputs.

In [18]:
log_message(trace, "Customer was identified as Premium 5G user.")
log_message(trace, "No citywide outage found; using SOP guidance.")

pprint(trace["logs"])

[{'message': 'Customer was identified as Premium 5G user.',
  'timestamp': '2026-03-25T06:38:52.343410'},
 {'message': 'No citywide outage found; using SOP guidance.',
  'timestamp': '2026-03-25T06:38:52.343473'}]


## Step 17 — Replay concept

Replay means: use the same input and inspect the same trace again, or re-run the same workflow with the same initial state.

Here we store a simple replay snapshot.

In [19]:
trace["replay_snapshot"] = {
    "initial_state": initial_state,
    "final_state": result,
}

pprint(trace["replay_snapshot"])

{'final_state': {'customer_id': 'CUST_1001',
                 'customer_profile': {'city': 'Pune',
                                      'customer_id': 'CUST_1001',
                                      'name': 'Ravi Patil',
                                      'network_type': '5G',
                                      'plan': 'Premium 5G',
                                      'recent_complaints': 2},
                 'diagnosis': 'If tower utilization exceeds 90 percent during '
                              'peak hours, classify as likely congestion and '
                              'notify NOC.',
                 'issue_type': 'slow_internet',
                 'next_action': 'Follow SOP guidance and monitor repeated '
                                'complaints.',
                 'outage_info': {'city': 'Pune',
                                 'issue_type': 'slow_internet',
                                 'outage_detected': False,
                                 'planned_mai

## Step 18 — Replay by re-running the same input

In a real debugging session, you may change one node or prompt and run again with the same input.

Here we simply re-run the graph with the same input to illustrate replay.

In [20]:
replay_trace = start_trace("telecom_incident_replay", initial_state)
TRACE_CONTEXT["current_trace"] = replay_trace

replay_result = graph.invoke(initial_state)

replay_trace["finished_at"] = datetime.utcnow().isoformat()
replay_trace["final_output"] = replay_result

print("Replay response:")
pprint(replay_result["response"])

Replay response:
{'completion_tokens': 45,
 'estimated_cost_usd': 0.0018,
 'prompt_tokens': 120,
 'response_text': 'Hello Ravi Patil, based on initial analysis for Pune, the '
                  'likely diagnosis is: If tower utilization exceeds 90 '
                  'percent during peak hours, classify as likely congestion '
                  'and notify NOC.. Next action: Follow SOP guidance and '
                  'monitor repeated complaints..'}


## Step 19 — Cost tracking

LangSmith can automatically record token usage and costs for major providers, and the docs also describe custom cost submission for additional components.

In this notebook, we simulated cost tracking by storing:
- prompt tokens
- completion tokens
- estimated cost in USD

In [21]:
print("Original trace cost summary:")
pprint(trace["cost_summary"])

print("\nReplay trace cost summary:")
pprint(replay_trace["cost_summary"])

Original trace cost summary:
{'completion_tokens': 45, 'estimated_cost_usd': 0.0018, 'prompt_tokens': 120}

Replay trace cost summary:
{'completion_tokens': 45, 'estimated_cost_usd': 0.0018, 'prompt_tokens': 120}


## Step 20 — What is LangSmith trace view?

LangSmith trace view helps you inspect:
- the full trace
- each nested run
- timing
- inputs / outputs
- errors
- metadata
- cost and token usage

For LangGraph applications, LangSmith has tracing support and docs show that setting environment variables can enable tracing. It also provides tracing quickstarts and a trace view in the UI for debugging execution steps.

## Step 21 — How to enable real LangSmith tracing

If you want to send real traces to LangSmith, the most common starting point is:

```python
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your_langsmith_api_key"
os.environ["LANGSMITH_PROJECT"] = "telecom-observability-lab"
```

Then run your LangGraph or LangChain workflow normally.

LangGraph docs note that tracing can be enabled by setting `LANGSMITH_TRACING=true` and the API key. LangSmith quickstart docs also describe enabling tracing and then viewing the traces in the LangSmith UI.

In [ ]:
# Example only - do not run unless you have real LangSmith credentials
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "your_langsmith_api_key"
# os.environ["LANGSMITH_PROJECT"] = "telecom-observability-lab"

## Step 22 — Mapping notebook concepts to LangSmith concepts

### In this notebook
- `trace` = a full workflow execution
- `runs` = node-level steps
- `logs` = custom messages
- `cost_summary` = cost tracking
- `replay_snapshot` = replay-friendly saved state

### In LangSmith
- trace = full end-to-end request
- run = one step / component inside trace
- trace view = UI to inspect runs and metadata
- cost tracking = token and price observability
- replay/debug = inspect and rerun workflows more confidently

## Step 23 — Telecom explanation for presentation

If you are teaching this topic, here is the simple story:

### Without observability
We only know the final answer:
- "Likely congestion"

### With observability
We know:
- which customer was looked up
- whether outage was checked
- which SOP was selected
- which diagnosis path was taken
- how much the run cost
- how to replay the same scenario

That is why observability is critical for enterprise telecom agents.

## Step 24 — Exercises

1. Add a `create_ticket` node and trace it.
2. Add a failure case where outage lookup returns an error and log that error.
3. Add a second telecom scenario for Mumbai packet loss and compare traces.
4. Add a `latency_ms` field to each run.
5. Add tags like `city=Pune` and `issue_type=slow_internet` to the trace metadata.

# Summary

This notebook demonstrated:

- LangGraph workflow construction
- MCP-style tool catalog thinking
- tool schema and contracts
- tracing concept
- logging concept
- replay concept
- cost tracking concept
- LangSmith trace-view style thinking
- how to enable real LangSmith tracing

This gives you a strong beginner foundation for **observability in telecom AI agents**.